# 变换与过渡

学习目标：能组合二维与三维变换，为状态变化添加可中断过渡，并解释显示隐藏过渡的起止状态。

前置知识：盒模型、正常流、层叠上下文、类选择器与交互伪类。

适用范围：CSS Transforms 与 Transitions 模块；独立变换属性及离散过渡需核对目标浏览器版本。页面不依赖 JavaScript。

工作目录：`content/Web与应用开发/css/`（以下命令从项目根目录切换到这里执行）。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/17-transforms-and-transitions/。

1. [index.html](scripts/17-transforms-and-transitions/index.html)、[transforms.css](scripts/17-transforms-and-transitions/transforms.css)：二维变换、顺序、原点与独立属性。
2. [three-d.html](scripts/17-transforms-and-transitions/three-d.html)、[three-d.css](scripts/17-transforms-and-transitions/three-d.css)：透视与保留三维空间。
3. [transitions.html](scripts/17-transforms-and-transitions/transitions.html)、[transitions.css](scripts/17-transforms-and-transitions/transitions.css)：交互过渡、缓动与中断。
4. [discrete.html](scripts/17-transforms-and-transitions/discrete.html)、[discrete.css](scripts/17-transforms-and-transitions/discrete.css)：离散过渡与起始样式。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/index.html)。

服务根目录是 content/Web与应用开发/css；修改文件后保存并刷新

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 移动外观，保留布局占位

transform 改变元素绘制时的坐标。translate(50%, 12px) 的两个值分别沿水平、垂直方向移动；页面坐标向右、向下为正。这里 50% 参照元素自身参考盒的宽度，本例普通 HTML 盒使用边框盒，不是父元素宽度。

变换不重新安排周围正常流内容，但可能扩大滚动溢出范围。非替换的普通行内盒不能直接变换；文字片段需要先成为 inline-block 等可变换盒。

非 none 的 transform 还会建立层叠上下文和后代定位的包含块，fixed 后代的参照也可能改变；不能把 translate(0) 当作完全没有 transform。

```html
<div class="track"><div class="tile moved">移动</div></div>
<p class="after">后面的段落仍按原占位排列。</p>
```

```css
.track { width: 220px; height: 90px; border: 1px dashed; }
.tile { box-sizing: border-box; width: 100px; height: 60px; background: #cce8ed; border: 2px solid #246; }
.moved { transform: translate(50%, 12px); }
/* 水平移动 50px：百分比相对自身参考盒宽度；track 的占位仍为原尺寸。 */
```

配套文件：[index.html](scripts/17-transforms-and-transitions/index.html)、[transforms.css](scripts/17-transforms-and-transitions/transforms.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| transform | 变换列表 | 组合变换函数 |
| translate | 独立平移 | 分别设置平移 |
| rotate | 独立旋转 | 设置角度与轴 |
| scale | 独立缩放 | 设置各轴比例 |
| transform-origin | 变换原点 | 旋转、缩放的参照点 |
| perspective | 透视距离 | 在父级设置子元素的透视 |
| perspective-origin | 透视原点 | 设置观察位置 |
| transform-style | 三维空间处理 | 控制后代扁平化或保留三维 |
| backface-visibility | 背面可见性 | 控制元素背面绘制 |
| transition | 过渡简写 | 设置各个过渡项目 |
| transition-property | 过渡属性 | 明确需要变化的属性 |
| transition-duration | 过渡时长 | 一次过渡持续时间 |
| transition-timing-function | 过渡缓动函数 | 时间与进度的关系 |
| transition-delay | 过渡延迟 | 推迟开始或从中途开始 |
| transition-behavior | 离散过渡行为 | 允许离散属性参与过渡 |

## 3 缩放、旋转与变换顺序

translateX(40px) 只沿水平轴移动；scale(1.5) 将两个轴的尺寸放大到 1.5 倍，1 表示原比例，两个数可以分别控制两轴。rotate(20deg) 使用角度；本例向右书写环境中正角度表现为顺时针。

transform 中的函数顺序有意义：矩阵按书写顺序组合，若跟踪一个点的最终位置，可从右侧函数向左计算。A 先缩放点再平移 40px；B 的平移也被左侧的缩放放大。不能把整串函数任意交换。

transform-origin 默认在普通 HTML 盒的中心。left top 把旋转、缩放的支点移到左上角；它不会把元素先布局到左上角。

```html
<div class="order-row"><div class="tile order-a">A</div></div>
<div class="order-row"><div class="tile order-b">B</div></div>
<div class="order-row"><div class="tile pivot">左上角旋转</div></div>
```

```css
.order-row { padding: 24px; height: 90px; border-bottom: 1px dotted; }
.order-a { transform-origin: 0 0; transform: translateX(40px) scale(1.5); }
.order-b { transform-origin: 0 0; transform: scale(1.5) translateX(40px); }
.pivot { transform-origin: left top; transform: rotate(20deg); }
/* A 的平移为 40px，B 的平移被缩放为 60px；两者宽度都变为 150px。 */
```

配套文件：[index.html](scripts/17-transforms-and-transitions/index.html)、[transforms.css](scripts/17-transforms-and-transitions/transforms.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/index.html)

## 4 独立变换属性与函数分开使用

translate、rotate、scale 是属性；translate()、rotate()、scale() 是 transform 值里的函数。独立属性允许分别覆盖一个动作，组合顺序固定为 translate → rotate → scale → transform，与声明书写顺序无关。

独立 translate 的两个值用空格分开；rotate 可写角度，三维时可写 y 35deg；scale 的一个数用于两轴。不要同时保留表达同一动作的 transform 与独立属性。

下例直接使用独立属性，要求浏览器支持这三项属性。故意逆序书写声明，观察组合顺序仍由规范固定。

```html
<div class="order-row"><div class="tile individual">独立属性</div></div>
```

```css
.individual {
  scale: 1.1;
  rotate: 10deg;
  translate: 40px;
  /* 声明顺序虽是缩放、旋转、平移，组合顺序仍为平移、旋转、缩放。 */
}
```

配套文件：[index.html](scripts/17-transforms-and-transitions/index.html)、[transforms.css](scripts/17-transforms-and-transitions/transforms.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/index.html)

## 5 三维变换与透视

三维增加 z 轴，正方向朝向观察者。rotateX()、rotateY() 分别绕水平轴、垂直轴旋转；translateZ() 沿深度轴移动，只接受长度，不能用百分比。

perspective 写在父元素上，让后代共享观察位置；数值是观察者到 z=0 平面的距离，正值越小，近大远小越明显。perspective-origin 调整观察位置。写在 transform 列表中的 perspective() 则参与该元素自身的变换顺序。

transform-style: preserve-3d 让子元素保留三维位置；默认 flat 会将后代压平到本元素平面。这一属性不继承，需要保持三维链的中间元素也要设置。opacity 小于 1、非 none 的 filter、overflow: hidden 等分组效果可能强制扁平化，不能只看 preserve-3d 声明是否存在。

backface-visibility: hidden 只控制背面绘制，不负责创建另一面内容。例子是静态空间对照；不要依靠翻到背面才能读到必要信息。

```html
<div class="scene">
  <div class="plane"><div class="face">向前 40px</div></div>
</div>
<div class="scene flat">
  <div class="plane"><div class="face">扁平化对照</div></div>
</div>
```

```css
.scene { width: 220px; height: 160px; margin: 30px; perspective: 500px; perspective-origin: center; }
.plane { width: 180px; height: 120px; border: 2px solid #246; transform: rotateY(35deg); transform-style: preserve-3d; }
.face { width: 140px; height: 80px; background: #cce8ed; transform: translateZ(40px); backface-visibility: hidden; }
.flat .plane { transform-style: flat; }
/* 比较 face 是否保留相对 plane 的深度；可临时把 rotateY 改为 180deg 观察背面。 */
```

配套文件：[three-d.html](scripts/17-transforms-and-transitions/three-d.html)、[three-d.css](scripts/17-transforms-and-transitions/three-d.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/three-d.html)

## 6 为状态变化设置过渡

transition 在旧值与新值之间生成中间值，需要前后状态、适当的动画类型和时间。这里只列 transform 与 background-color，避免 all 让以后新增的尺寸变化也意外过渡。

每个逗号分隔项目是一套配置：第一个时间是持续时间，第二个是延迟；800ms 等于 0.8s。时长不能为负；正延迟先等待，负延迟表示立即从已进行一段时间的位置开始。省略时长的默认值为 0s，通常看不到平滑变化。

把 transition 放在基础状态上，进入和离开都能使用。:hover 选择悬停状态，:focus-visible 为键盘焦点提供同类反馈；若仍有焦点，仅移走指针未必回程。

过渡中途出现新的目标，会从当前表现值重新朝目标变化；返回原状态时规范有反向缩短规则，不能承诺回程仍耗满 800ms。颜色、数值或匹配的变换可以插值，但能否平滑变化取决于具体属性和值；不要假定 auto 到长度总能在传统过渡里插值。

```html
<button class="action" type="button">悬停或 Tab 聚焦</button>
<p>移开指针并移走焦点，可观察回程。</p>
```

```css
.action {
  padding: 12px 20px;
  background: #d8edf2;
  transform: translateX(0);
  transition: transform 800ms ease-out 100ms, background-color 400ms linear;
}
.action:hover, .action:focus-visible { transform: translateX(40px); background-color: #f6d69b; }
/* 在 800ms 内移出且移走焦点：观察过渡从当前值反向，不跳回起点再播放。 */
```

配套文件：[transitions.html](scripts/17-transforms-and-transitions/transitions.html)、[transitions.css](scripts/17-transforms-and-transitions/transitions.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/transitions.html)

## 7 缓动与减少运动偏好

缓动（easing）把已经过去的时间比例映射为动画进度。linear 匀速，ease、ease-in、ease-out、ease-in-out 分别使用预定义曲线。cubic-bezier() 的四个数依次是两个控制点的横纵坐标；横坐标必须在 0 到 1 内，纵坐标可越界，因而可能产生超出端点的进度。

steps(4, end) 把过程分成四步，在每步末尾跳跃；分步缓动不等于属性本身是离散动画类型。多个过渡分项用列表对应，较短的缓动列表会重复使用。

示例末尾在 prefers-reduced-motion: reduce 下设置 transition: none，状态仍立即更新。这个媒体特性表达减少非必要运动的偏好，不是让按钮停止响应。

```html
<input id="move" type="checkbox"><label for="move">比较缓动</label>
<div class="lane"><div class="dot linear">匀速</div></div>
<div class="lane"><div class="dot bezier">曲线</div></div>
<div class="lane"><div class="dot stepped">四步</div></div>
```

```css
.lane { width: 260px; height: 50px; border-bottom: 1px dotted; }
.dot { width: 80px; background: #cce8ed; transition: transform 2s linear; }
.bezier { transition-timing-function: cubic-bezier(0.2, 0.8, 0.3, 1); }
.stepped { transition-timing-function: steps(4, end); }
#move:checked ~ .lane .dot { transform: translateX(160px); }
/* 三者时长和终点相同；四步示例每次跳跃 40px，曲线只改变时间到进度的映射。 */
@media (prefers-reduced-motion: reduce) {
  .action, .dot, .panel { transition: none; }
}
```

配套文件：[transitions.html](scripts/17-transforms-and-transitions/transitions.html)、[transitions.css](scripts/17-transforms-and-transitions/transitions.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/transitions.html)

## 8 离散过渡与 @starting-style

离散（discrete）变化通常只在两个值之间切换，不生成连续中间值。transition-behavior: allow-discrete 允许这类属性启动过渡；大多数离散值在进度中点切换，display 从 none 进入时在开头切换、退出到 none 时在结尾切换，使淡入淡出期间内容可见。

传统过渡通常没有首次显示或 display: none 之后的“之前样式”。@starting-style 为这种情况提供起始声明；它是 @ 规则，不是属性或关键帧。这里放在最终状态之后，避免同等优先级下起始 opacity 被后面的最终值覆盖。

本例要求浏览器同时支持 CSS Transitions Level 2 的离散 display 过渡和 @starting-style。直接声明这两种机制，分别观察进入起点和退出时机；减少运动时保留立即切换。

display 切换会改变布局；淡出期间内容仍存在，实际交互界面还需根据业务管理焦点。这里面板只有文字，没有需要退出管理的可聚焦控件。

```html
<input id="show" type="checkbox"><label for="show">显示说明</label>
<div class="panel"><p>这段内容可立即切换，也可渐入渐出。</p></div>
<p>后续内容用于观察显示隐藏时的重新排版。</p>
```

```css
.panel { display: none; opacity: 0; padding: 16px; background: #d8edf2; }
#show:checked ~ .panel { display: block; opacity: 1; }
.panel { transition: opacity 600ms, display 600ms allow-discrete; }
@starting-style {
  #show:checked ~ .panel { opacity: 0; }
}
/* 支持完整组合时：进入之初 display 变为 block，退出结束才变为 none。 */
```

配套文件：[discrete.html](scripts/17-transforms-and-transitions/discrete.html)、[discrete.css](scripts/17-transforms-and-transitions/discrete.css) · [浏览器预览](http://127.0.0.1:8101/scripts/17-transforms-and-transitions/discrete.html)

## 本章小结

- 变换改变外观和坐标关系，正常流占位保留，但溢出、包含块和层叠可能改变。
- transform 函数顺序与独立属性固定顺序需要分别理解；原点和透视位置也不同。
- 过渡连接两个状态，缓动控制过程，中断可改变后续时长。
- 显示隐藏过渡要区分进入起始样式与退出末尾的display切换；减少运动时状态仍立即更新。

## 练习

在 scripts/17-transforms-and-transitions/ 内修改少量规则，完成后恢复原文件。

（1）把移动例子的宽度改为 120px，预测 50% 的平移量。检查元素计算宽度与可见偏移，确认后续段落的布局位置没有随平移改变。

（2）交换 A 的函数顺序并改为左上角缩放 2 倍，先预测平移再用盒子位置核对；不要只比较截图大小。

（3）把过渡延迟改为 500ms，在延迟内取消状态，再在运动中取消状态，比较两者；启用减少运动后确认状态仍可切换。

（4）只临时禁用 discrete.css 中的 @starting-style，保留transition与状态规则。比较进入和退出是否仍淡变，再恢复起始规则。

### 提示

百分比从被变换元素参考盒出发；过渡状态同时检查悬停和焦点。开发者工具中的临时修改需另行保存才会保留。

### 重点题解析

第（2）题只将A改为transform:scale(2) translateX(40px)，保持原点0 0及100px宽度。点先平移40px，再整体乘2，因此可见平移80px、宽200px；B保持原例的60px和150px。

第（4）题缺少starting-style时，none后的首次显示没有淡入起点，进入直接可见；退出已有之前样式，opacity仍可向0过渡，display在退出结束才none。恢复starting-style后，进入也从opacity:0淡入；两种情形内容均完整。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| W3C | [CSS Transforms 1 §3–5](https://www.w3.org/TR/css-transforms-1/#transform-rendering) 的布局、包含块、变换列表与原点；[CSS Transforms 2 §5–10](https://www.w3.org/TR/css-transforms-2/#individual-transforms) 的独立属性、三维空间、分组扁平化与透视；[CSS Transitions 1 §2–3.1](https://www.w3.org/TR/css-transitions-1/#transitions) 的过渡参数、启动和中断反向；[CSS Transitions 2 §3.3](https://www.w3.org/TR/css-transitions-2/#defining-before-change-style) 的首次显示起始样式。 |
| MDN | [transform](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/transform)、[translate](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/translate)、[rotate](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/rotate)、[scale](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/scale) 的语法与适用对象；[perspective](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/perspective)、[transform-style](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/transform-style#description) 的空间关系与扁平化条件；[transition](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/transition)、[transition-timing-function](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/transition-timing-function)、[cubic-bezier()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/easing-function/cubic-bezier) 的分项与缓动；[transition-behavior](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/transition-behavior#discrete_animation_behavior)、[@starting-style](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@starting-style#description) 的离散切换、起始状态及各页 Browser compatibility；[prefers-reduced-motion](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@media/prefers-reduced-motion) 的用户偏好。 |
| Python 3.12 | [http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的服务目录、端口与绑定地址。 |